In [0]:
%sql
use catalog `e-commerce`;
use gold;

In [0]:
#combined table for kpi
orders = spark.table("gold.fact_orders")
items = spark.table("gold.fact_order_items")
reviews = spark.table("gold.fact_reviews")
customers = spark.table("gold.dim_customer")
products = spark.table("gold.dim_product")
sellers = spark.table("gold.dim_seller")

kpi_table = orders.join(items, "order_id").join(customers, "customer_id").join(products, "product_id").join(sellers, "seller_id").join(reviews, "order_id", "left")

kpi_table.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.kpi_table")

In [0]:
display(kpi_table)

In [0]:
#total revenue by month
from pyspark.sql.functions import sum, month, year

total_revenue_by_month=kpi_table.groupBy(
    year("order_date").alias("year"),
    month("order_date").alias("month")
).agg(
    sum("total_payment").alias("total_revenue")
).orderBy("year", "month")

total_revenue_by_month.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.total_revenue_by_month")

In [0]:
#avg order value
from pyspark.sql.functions import countDistinct,desc

avg_order_val=kpi_table.select(
    (sum("total_payment") / countDistinct("order_id")).alias("AOV")
)
avg_order_val.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.avg_order_val")

In [0]:
#top products
top_products=kpi_table.groupBy("product_id").agg(sum("total_payment").alias("revenue")).orderBy(desc("revenue")).limit(10)

top_products.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.top_products")

In [0]:
#top sellers
top_sellers=kpi_table.groupBy("seller_id").agg(sum("price").alias("revenue")).orderBy(desc("revenue")).limit(10)

top_sellers.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.top_sellers")

In [0]:
#orders by location
orders_by_loc=kpi_table.groupBy("customer_city", "customer_state").agg(countDistinct("order_id").alias("total_orders")).orderBy(desc("total_orders"))

orders_by_loc.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.orders_by_loc")

In [0]:
#new vs returning customers monthly

from pyspark.sql.window import Window
from pyspark.sql.functions import min,when,col

# First order date per customer
window_spec = Window.partitionBy("customer_unique_id")

df = kpi_table.withColumn(
    "first_order_date",
    min("order_date").over(window_spec)
)

df = df.withColumn(
    "customer_type",
    when(col("order_date") == col("first_order_date"), "new")
    .otherwise("returning")
)

customer_dist=df.groupBy("customer_type") \
    .agg(countDistinct("customer_unique_id").alias("count"))
customer_dist.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.customer_dist")

In [0]:
#customer lifetime value
kpi_table.groupBy("customer_unique_id").agg(sum("total_payment").alias("CLV")).orderBy(desc("CLV")).display()

In [0]:
#order fulfillment rate
order_fulfillment_rate=kpi_table.select(
    (sum("is_delivered") / countDistinct("order_id") * 100)
    .alias("fulfillment_rate")
)
order_fulfillment_rate.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.order_fulfillment_rate")

In [0]:
#late delivery % 
from pyspark.sql.functions import count,sum
late_delivery_pct=kpi_table.select(
    (sum("is_late") / count("*") * 100).alias("late_pct")
)
late_delivery_pct.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.late_delivery_pct")

In [0]:
#avg delivery time
from pyspark.sql.functions import avg
avg_delivery_time=kpi_table.select(
    avg("delivery_days").alias("avg_delivery_days")
)
avg_delivery_time.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.avg_delivery_time")

In [0]:
#payment method distribution
from pyspark.sql.functions import count
payments = spark.table("gold.fact_payments")

payment_method_dist=payments.groupBy("payment_type").agg(count("*").alias("count")).orderBy(desc("count"))
payment_method_dist.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.payment_method_dist")

In [0]:
#avg review score per product category
from pyspark.sql.functions import avg
avg_review_score_per_prod_cat=kpi_table.groupBy("product_category_final").agg(avg("review_score").alias("avg_review_score")).orderBy(desc("avg_review_score"))
avg_review_score_per_prod_cat.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.avg_review_score_per_prod_cat")


In [0]:
#avg review score by seller
avg_review_per_seller=kpi_table.groupBy("seller_id").agg(avg("review_score").alias("avg_review_score")).orderBy(desc("avg_review_score")).limit(10)
avg_review_per_seller.write.format("delta").mode("overwrite").option("overwriteSchema","true").saveAsTable("gold.avg_review_per_seller")


In [0]:
display(payment_method_dist)